## <font color='blue'>Fine-Tuning de LLM Open-Source Para App de Assistente</font>

## Instalando e Carregando Pacotes

In [ ]:
# 1. Descarregar e instalar o Miniconda silenciosamente no Kaggle
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /opt/conda -u

# 2. Adicionar o Conda ao PATH do sistema atual
import os
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

# NOVO: Aceitar os Termos de Serviço da Anaconda exigidos pelo instalador
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# 3. Criar o ambiente isolado com o Python 3.10
!conda create -n py310 python=3.10 -y

# Instalando os pacotes
!/opt/conda/envs/py310/bin/pip install -q -r /kaggle/input/datasets/danielrrezende/datasetdsa/cap9/requirements.txt

# 4. Instalar o PyTorch usando o pip específico do novo ambiente (caminho absoluto)
# !/opt/conda/envs/py310/bin/pip install -q torch==2.1.0 --index-url https://download.pytorch.org/whl/cu121

# 5. Instalar o Numpy (binário) e as restantes bibliotecas
# !/opt/conda/envs/py310/bin/pip install -q numpy==1.25.2 --only-binary :all:
# !/opt/conda/envs/py310/bin/pip install -q nltk==3.8.1 evaluate==0.4.1 datasets==2.17.0 transformers==4.37.2 ipykernel

# 6. Registar o novo kernel no Jupyter
!/opt/conda/envs/py310/bin/python -m ipykernel install --user --name py310 --display-name "Python 3.10 (Compatibilidade)"

In [9]:
# Imports
import numpy as np
import nltk
import shutil
import evaluate
import transformers
import datasets
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

RuntimeError: Failed to import transformers.trainer_seq2seq because of the following error (look up to see its traceback):
cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
# Remove as pastas, se já existirem. Isso evita erros ao executar o jupyter a partir da segunda vez em diante
try:
    shutil.rmtree('logs_treino')
    shutil.rmtree('resultados_treino')
    shutil.rmtree('modelo_salvo')
except:
    print('As pastas não existem ou já foram removidas!')

## Carregando os Dados

In [ ]:
# Define o nome do arquivo
nome_arquivo = "dataset.csv"

In [ ]:
# Carrega os dados
ds_dataset = load_dataset('csv', data_files = nome_arquivo, delimiter = ',')

In [ ]:
# Divisão em treino e teste com proporção 80/20
ds_dataset = ds_dataset["train"].train_test_split(test_size = 0.2)

In [ ]:
# Visualiza o formato do dataset
ds_dataset

## Tokenizador e LLM Open-Source

https://huggingface.co/google/flan-t5-base

In [ ]:
# Carrega o tokenizador
tokenizador = T5Tokenizer.from_pretrained("google/flan-t5-base", legacy = False)

In [ ]:
# Visualiza o tokenizador
tokenizador

In [ ]:
# Carrega o LLM pré-treinado
modelo = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

In [ ]:
# Visualiza a arquitetura do LLM
modelo

In [ ]:
# Data collator para concatenar tokenizador e modelo
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizador, model = modelo)

In [ ]:
# Visualiza o data collator
data_collator

## Pré-Processamento

In [ ]:
# Todos os inputs vão receber como prefixo: "answer the question" (responda a questão)
prefix = "answer the question: "

In [ ]:
# Função de pré-processamento
def ds_fn_preprocess(data):

    # Concatena o prefixo a cada pergunta na lista de perguntas fornecida em data["question"]
    inputs = [prefix + doc for doc in data["question"]]

    # Usa o tokenizer para converter as perguntas processadas em tokens com um comprimento máximo de 128,
    # truncando as que forem mais longas
    model_inputs = tokenizador(inputs, max_length = 128, truncation = True)

    # Tokeniza as respostas fornecidas em data["answer"] com um comprimento máximo de 512,
    # truncando as que forem mais longas
    labels = tokenizador(text_target = data["answer"], max_length = 512, truncation = True)

    # Adiciona os tokens das respostas como rótulos no dicionário de entradas do modelo
    model_inputs["labels"] = labels["input_ids"]

    # Retorna o dicionário processado contendo tanto as entradas tokenizadas quanto os rótulos
    return model_inputs

In [ ]:
# Aplica a função de pré-processamento ao dataset gerando o dataset tokenizado
ds_dataset_tokenized = ds_dataset.map(ds_fn_preprocess, batched = True)

In [ ]:
# Visualiza o dataset tokenizado
ds_dataset_tokenized

In [ ]:
ds_dataset_tokenized['train']['question'][0]

In [ ]:
ds_dataset_tokenized['train']['answer'][0]

In [ ]:
ds_dataset_tokenized['train']['input_ids'][0]

In [ ]:
ds_dataset_tokenized['train']['labels'][0]

## Definindo a Métrica de Avaliação do Modelo

In [ ]:
# O pacote "punkt" é específico para a tarefa de tokenização, que envolve a divisão de um texto
# em uma lista de sentenças
nltk.download("punkt", quiet = True)

In [ ]:
# Define a métrica
metric = evaluate.load("rouge")

In [ ]:
# Função de cálculo das métricas
def ds_calcula_metricas(eval_preds):

    # Desempacota as predições e os rótulos do argumento eval_preds
    previsoes, labels = eval_preds

    # Substitui todos os valores diferentes de -100 em labels pelo ID do token de preenchimento
    labels = np.where(labels != -100,
                      labels,
                      tokenizador.pad_token_id)

    # Decodifica as previsões para texto, ignorando tokens especiais
    previsoes_decodificadas = tokenizador.batch_decode(previsoes,
                                                       skip_special_tokens = True)

    # Decodifica os labels para texto, ignorando tokens especiais
    labels_decodificados = tokenizador.batch_decode(labels,
                                                    skip_special_tokens = True)

    # Adiciona uma nova linha após cada sentença para as previsões decodificadas,
    # preparando-as para a avaliação ROUGE
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in previsoes_decodificadas]

    # Adiciona uma nova linha após cada label para as previsões decodificadas,
    # preparando-os para a avaliação ROUGE
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in labels_decodificados]

    # Calcula a métrica ROUGE entre as previções e os rótulos decodificados, utilizando um stemmer
    resultado = metric.compute(predictions = previsoes_decodificadas,
                               references = labels_decodificados,
                               use_stemmer = True)

    # Retorna o resultado da métrica ROUGE
    return resultado

In [ ]:
# Define argumentos de treino
ds_training_args = Seq2SeqTrainingArguments(output_dir = "resultados_treino",
                                             evaluation_strategy = "epoch",
                                             learning_rate = 3e-4,
                                             logging_dir = "logs_treino",
                                             logging_steps = 1,
                                             per_device_train_batch_size = 4,
                                             per_device_eval_batch_size = 2,
                                             weight_decay = 0.01,
                                             save_total_limit = 3,
                                             num_train_epochs = 3,
                                             predict_with_generate = True,
                                             push_to_hub = False)

In [ ]:
# Define o trainer
ds_trainer = Seq2SeqTrainer(model = modelo,
                             args = ds_training_args,
                             train_dataset = ds_dataset_tokenized["train"],
                             eval_dataset = ds_dataset_tokenized["test"],
                             tokenizer = tokenizador,
                             data_collator = data_collator,
                             compute_metrics = ds_calcula_metricas)

## Treinamento do Modelo

In [ ]:
%%time
ds_trainer.train()

In [ ]:
# Salva o modelo
ds_trainer.save_model('modelo_salvo')

**ROUGE-1** mede a sobreposição de unigramas (palavras individuais).

**ROUGE-2** mede a sobreposição de bigramas (pares de palavras consecutivas).

**ROUGE-L** mede a sobreposição da subsequência mais longa comum entre o resumo gerado e o resumo de referência. Isso leva em conta a ordem das palavras, mas permite lacunas. Valores mais altos indicam melhor desempenho. ROUGE-L é calculado com base na similaridade entre as sequências, levando em consideração a precisão, o recall e a média harmônica entre eles.

Valores mais altos de ROUGE indicam uma maior semelhança entre o resumo gerado e o resumo de referência, o que geralmente é interpretado como uma indicação de melhor qualidade do resumo. No entanto, é importante lembrar que nenhuma métrica pode capturar completamente a qualidade de um resumo e é útil complementar a avaliação com análises qualitativas ou outras métricas.

## Deploy e Uso do Modelo

In [ ]:
# Carrega o tokenizador final salvo em disco
tokenizador_ds_final = AutoTokenizer.from_pretrained('modelo_salvo')

In [ ]:
# Carrega o modelo final salvo em disco
modelo_ds_final = AutoModelForSeq2SeqLM.from_pretrained('modelo_salvo')

In [ ]:
# Visualiza a arquitetura do modelo
modelo_ds_final

In [ ]:
# Texto de entrada
texto_entrada = "Can I move out of state with my children if I have a custody agreement in that state?"

In [ ]:
# Tokeniza o input
texto_entrada_tokenizado = tokenizador_ds_final(texto_entrada, return_tensors = "pt").input_ids

In [ ]:
# Gera a saída (previsão do modelo)
texto_saida_tokenizado = modelo_ds_final.generate(texto_entrada_tokenizado,
                                                   max_length = 50,
                                                   temperature = 0.4,
                                                   do_sample = True)

In [ ]:
# Visualiza
texto_saida_tokenizado

In [ ]:
# Decode da saída
texto_saida = tokenizador_ds_final.decode(texto_saida_tokenizado[0],
                                           skip_special_tokens = True)

In [ ]:
# Resultado
print("Pergunta:", texto_entrada)
print("Resposta:", texto_saida)

In [ ]:
#%watermark -v -m

In [ ]:
#%watermark --iversions

# Fim